[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/03_AB%E3%83%86%E3%82%B9%E3%83%88%E3%81%A8KPI.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 実践マーケ-03. A/Bテストで意思決定する

**舞台設定**：申込ページのボタンの色を「青(A)」と「緑(B)」で出し分けた。
どちらが申込率が高いか、`ab_test.csv` で科学的に判断します。

**使う統計（3〜2級）**：比率の比較・カイ二乗検定・「差は偶然か？」の判断

In [ ]:
import pandas as pd              # 表データ
import numpy as np               # 数値計算
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
ab = pd.read_csv('../data/ab_test.csv')           # A/Bテストデータを読み込む
ab.head()                        # 先頭5行を確認

### 📋 使うデータ：`ab_test.csv`（LPボタン色のA/Bテスト）
1行＝訪問1件。`申込`は 1=申込んだ / 0=申込まず。グループは A_青ボタン / B_緑ボタン の2群（各2400件前後）。

| グループ | 申込 |
|---|---|
| B_緑ボタン | 0 |
| A_青ボタン | 0 |
| B_緑ボタン | 1 |

## 1. 各グループの申込率を出す

まず単純に「見た目の率」を比べます。

In [ ]:
summary = ab.groupby('グループ')['申込'].agg(['count','sum','mean'])  # 群ごとに件数・申込数・申込率
summary.columns = ['訪問数','申込数','申込率']   # 列名を分かりやすく
summary['申込率'] = (summary['申込率']*100).round(2)  # 割合を%に
summary

In [ ]:
summary['申込率'].plot(kind='bar', figsize=(5,4), title='ボタン色別の申込率(%)')  # 群別の申込率を棒グラフに
plt.ylabel('申込率(%)'); plt.xticks(rotation=0); plt.show()

## 2. その差は「偶然」ではない？

Bの方が高く見えても、たまたまかもしれません。
**カイ二乗検定**で「色と申込に関連があるか」を確かめます。

- H₀（帰無仮説）：色と申込率は関係ない（差は偶然）
- H₁（対立仮説）：色によって申込率が違う

In [ ]:
table = pd.crosstab(ab['グループ'], ab['申込'])   # グループ×申込のクロス表
print(table)
chi2, p, dof, exp = stats.chi2_contingency(table)  # カイ二乗検定（差が偶然か）
print(f'\nカイ二乗={chi2:.3f}, p値={p:.4f}')
if p < 0.05:
    print('→ 有意差あり！ ボタンの色で申込率は本当に変わる')
else:
    print('→ 有意差なし。偶然の範囲かもしれない')

## 3. 効果の大きさと事業インパクト

「統計的に有意」だけでなく「ビジネスとして何円得か」も大事です。

In [ ]:
rate_a = summary.loc['A_青ボタン','申込率'] / 100   # Aの申込率（割合に戻す）
rate_b = summary.loc['B_緑ボタン','申込率'] / 100   # Bの申込率
lift = (rate_b - rate_a) / rate_a * 100             # 改善率（リフト）
print(f'改善率（リフト）: {lift:+.1f}%')

# 月10万訪問・1申込5000円の価値と仮定したら？
monthly_visits = 100000                    # 月間訪問数
value_per = 5000                           # 1申込の価値
gain = monthly_visits * (rate_b - rate_a) * value_per   # 増加見込み額
print(f'Bに変えると月あたり約 {gain:,.0f} 円の増加見込み')

> 📝 **A/Bテストの注意点**
> - 途中で何度も覗いて「有意になった瞬間にやめる」のはNG（偽陽性が増える）
> - 事前に必要なサンプル数を決めておく
> - 期間は曜日の偏りをならすため1週間以上が目安

---
## 🏆 ワークシート課題

**課題1.** AとBの申込率の差を「パーセントポイント」と「リフト%」の両方で表そう。

**課題2.** もし有意水準を 0.01（より厳しく）にしても結論は変わるか、p値を見て判断しよう。

**課題3.**（意思決定）上司に「緑ボタンを採用すべきか」を、
統計的根拠（p値）とビジネス的根拠（増加見込み金額）の両方で説明する文を書こう。

**課題4.**（発展）`proportions_ztest` を使って同じ検定を比率の検定で行い、
カイ二乗検定とp値がほぼ一致することを確かめよう。

In [ ]:
# 課題1


In [ ]:
# 課題4（ヒント）
from statsmodels.stats.proportion import proportions_ztest  # 母比率の差の検定

> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[03_ABテストとKPI の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/03_AB%E3%83%86%E3%82%B9%E3%83%88%E3%81%A8KPI.md)**

🎉 **実践マーケ編 クリア！** 統計が「ビジネスの意思決定の道具」だと実感できたはず。
ひと息ついたら `06_ホビー_陶芸3D` で 3Dモデリングを楽しもう。